In [2]:
import json
import os
import math
from IPython.display import display, clear_output

import pandas as pd
import numpy as np
import pickle 

import matplotlib.pyplot as plt


pd.set_option('display.max_columns', None)

In [3]:
df_table = pd.read_csv("elo_ratings.csv")

In [4]:
df_table[["team", "total", "goals_for", "goals_against"]]

,team,total,goals_for,goals_against
0,Spain,780,1591,697
1,Argentina,1109,2112,1136
2,France,938,1706,1272
3,England,1160,2719,1118
4,Brazil,1065,2294,954
...,...,...,...,...
239,Niue,2,0,33
240,Northern Mariana Islands,36,43,153
241,Cocos Islands,10,8,29
242,Palau,2,3,27


In [5]:
# Offensive/defensive multipliers derived from Elo (opponent-strength adjusted).
# Centered at 1500 with the usual 400-point scale so an average team is ~1.0 on both axes.
ELO_BASE = 1500
ELO_SCALE = 400

elo_delta = (df_table["rating"] - ELO_BASE) / ELO_SCALE
df_table["offensive_rating"] = np.exp(elo_delta)
df_table["defensive_rating"] = np.exp(-elo_delta)

# Keep per-game goal stats for reference / inspection.
df_table["goals_for_per_game"] = df_table["goals_for"] / df_table["total"]
df_table["goals_against_per_game"] = df_table["goals_against"] / df_table["total"]


In [19]:
df_table[["team", "total", "goals_for", "goals_against", "goals_for_per_game", "goals_against_per_game", "offensive_rating", "defensive_rating"]].sort_values("offensive_rating", ascending=False)[:50]

,team,total,goals_for,goals_against,goals_for_per_game,goals_against_per_game,offensive_rating,defensive_rating
221,Saba,8,25,26,3.125000,3.250000,2.347645,1.591089
94,Kurdistan,4,12,3,3.000000,0.750000,2.253739,0.367174
212,Sint Eustatius,11,33,30,3.000000,2.727273,2.253739,1.335180
233,Christmas Island,10,29,8,2.900000,0.800000,2.178615,0.391653
159,Tahiti,254,693,383,2.728346,1.507874,2.049661,0.738204
141,New Caledonia,285,739,419,2.592982,1.470175,1.947969,0.719748
217,Saint Barthelemy,10,25,29,2.500000,2.900000,1.878116,1.419741
3,England,1160,2719,1118,2.343966,0.963793,1.760896,0.471840
172,Fiji,286,644,459,2.251748,1.604895,1.691618,0.785702
10,Germany,1045,2352,1232,2.250718,1.178947,1.690844,0.577172


In [7]:
df_fixtures = pd.read_csv("worldcup_2026_fixtures.csv")

In [8]:
df_fixtures[:72]

,match_number,round,date,time,team1,team2,group,venue
0,1,Matchday 1,2026-06-11,13:00 UTC-6,Mexico,South Africa,Group A,Mexico City
1,2,Matchday 1,2026-06-11,20:00 UTC-6,South Korea,Czech Republic,Group A,Guadalajara (Zapopan)
2,7,Matchday 2,2026-06-12,15:00 UTC-4,Canada,Bosnia & Herzegovina,Group B,Toronto
3,19,Matchday 2,2026-06-12,18:00 UTC-7,USA,Paraguay,Group D,Los Angeles (Inglewood)
4,8,Matchday 3,2026-06-13,12:00 UTC-7,Qatar,Switzerland,Group B,San Francisco Bay Area (Santa Clara)
...,...,...,...,...,...,...,...,...
67,72,Matchday 17,2026-06-27,17:00 UTC-4,Croatia,Ghana,Group L,Philadelphia
68,65,Matchday 17,2026-06-27,19:30 UTC-4,Colombia,Portugal,Group K,Miami (Miami Gardens)
69,66,Matchday 17,2026-06-27,19:30 UTC-4,DR Congo,Uzbekistan,Group K,Atlanta
70,59,Matchday 17,2026-06-27,21:00 UTC-5,Algeria,Austria,Group J,Kansas City


In [9]:
table_teams = (set(df_table.team))

In [10]:
fixtures_teams = (set(df_fixtures[:72].team1).union(set(df_fixtures[:72].team2)))

In [11]:
len(fixtures_teams.intersection(table_teams))

45

In [12]:
fixtures_teams - fixtures_teams.intersection(table_teams)

{'Bosnia & Herzegovina', 'Czech Republic', 'USA'}

In [13]:
df_fixtures = df_fixtures.replace("Bosnia & Herzegovina", "Bosnia and Herzegovina")

In [14]:
df_fixtures = df_fixtures.replace("Czech Republic", "Czechia")

In [15]:
df_fixtures = df_fixtures.replace("USA", "United States")

In [16]:
df_ratings = df_table[df_table.team.isin(df_fixtures.team1) | df_table.team.isin(df_fixtures.team2)]

In [17]:
df_ratings = df_ratings[["team", "rank", "rating", "total", "goals_for", "goals_against", "offensive_rating", "defensive_rating"]]

# Normalize so tournament averages are 1.0 — keeps the 1.25 gpg baseline meaningful.
df_ratings["offensive_rating"] /= df_ratings["offensive_rating"].mean()
df_ratings["defensive_rating"] /= df_ratings["defensive_rating"].mean()

In [18]:
df_ratings.sort_values("offensive_rating", ascending=False)

,team,rank,rating,total,goals_for,goals_against,offensive_rating,defensive_rating
3,England,4,2020,1160,2719,1118,1.760896,0.471840
10,Germany,11,1923,1045,2352,1232,1.690844,0.577172
4,Brazil,5,1984,1065,2294,954,1.618178,0.438541
7,Netherlands,8,1961,901,1892,1142,1.577534,0.620515
0,Spain,1,2165,780,1591,697,1.532350,0.437471
25,Australia,26,1775,636,1289,715,1.522573,0.550377
41,Sweden,42,1719,1118,2226,1482,1.495773,0.648959
1,Argentina,2,2113,1109,2112,1136,1.430688,0.501485
31,Iran,32,1764,669,1273,525,1.429502,0.384188
36,Czechia,37,1733,883,1626,1098,1.383382,0.608769


In [20]:
df_ratings.to_csv("teams.csv")

In [230]:
strength_pl = get_strength_stats(premier_league)
strength_ch = get_strength_stats(championship, league_correction=0.4)

In [231]:
strength = {}

for stat_type in strength_pl.keys():
    strength[stat_type] = pd.concat([strength_pl[stat_type][:17], strength_ch[stat_type][strength_ch[stat_type].Squad.isin(promoted_teams)]], ignore_index=True)

In [423]:
teams = sorted(['Manchester City',
 'Arsenal',
 'Liverpool',
 'Aston Villa',
 'Tottenham',
 'Chelsea',
 'Newcastle Utd',
 'Manchester Utd',
 'West Ham',
 'Crystal Palace',
 'Brighton',
 'Bournemouth',
 'Fulham',
 'Wolves',
 'Everton',
 'Brentford',
 "Nott'ham Forest",
 'Leicester City',
 'Ipswich Town',
 'Southampton'])

In [474]:
home_sw_start = strength["home"][["Squad", "attacking_strength", "defensive_weakness"]].set_index("Squad").to_dict(orient="index")
away_sw_start = strength["away"][["Squad", "attacking_strength", "defensive_weakness"]].set_index("Squad").to_dict(orient="index")

In [475]:
def get_new_league_table():
    league_table = {}
    for table_type in ["summary", "home", "away"]:
        league_table[table_type] = pd.DataFrame({
    "Squad": teams,
    "MP": [0]*20, 
    "W": [0]*20,
    "D": [0]*20,
    "L": [0]*20,
    "GF": [0]*20,
    "GA": [0]*20,
    "GD": [0]*20,
    "Pts": [0]*20,
}).set_index("Squad")
    return league_table

In [577]:
def simulate_matchweek(df_fixtures, week, home_strength, away_strength, elo, GF, GA):
    matchweek = []
    fixtures = df_fixtures[df_fixtures.Wk == week]
    for fixture in fixtures.itertuples():
        home_team = fixture.Home
        away_team = fixture.Away

        elo_factor = 1/(1+10**((elo[away_team]-elo[home_team])/400))

        #Home team attack strength * away team defence strength * average number of home goals
        lambda_home = home_strength[home_team]["attacking_strength"]*away_strength[away_team]["defensive_weakness"]*GF#*elo_factor
        lambda_away = away_strength[away_team]["attacking_strength"]*home_strength[home_team]["defensive_weakness"]*GA#*(1-elo_factor)

        simulated_home_goals = np.random.poisson(lam=lambda_home)
        simulated_away_goals = np.random.poisson(lam=lambda_away)

        simulated_home_goals, simulated_away_goals

        gd = simulated_home_goals - simulated_away_goals

        elo[home_team] += 40 * ((1 if gd > 0 else 0.5 if gd == 0 else 0) - elo_factor)
        elo[away_team] += 40 * ((1 if gd < 0 else 0.5 if gd == 0 else 0) - (1-elo_factor))
        
        matchweek.append(
            {
                "week": week,
                "home": home_team,
                "away": away_team,
                "home_goals": simulated_home_goals,
                "away_goals": simulated_away_goals,
            }
        )
        
    return matchweek, elo

In [586]:
def update_sw(league_table, home_sw, away_sw, GF, GA):
    home_sw = pd.DataFrame.from_dict(home_sw, orient="index")
    away_sw = pd.DataFrame.from_dict(away_sw, orient="index")
    home_update = (
        pd.concat([
            (home_sw.attacking_strength * 19 + league_table["home"].GF * league_table["home"].MP / league_table["home"].GF.mean())/(19+league_table["home"].MP), 
            (home_sw.defensive_weakness * 19 + league_table["home"].GA * league_table["home"].MP / league_table["home"].GA.mean())/(19+league_table["home"].MP)
        ], 
        axis=1
        ).rename(
        columns={
            0: "attacking_strength",
            1: "defensive_weakness"
        }
        )
    ).to_dict(orient="index")
    
    away_update = (
        pd.concat([
            (away_sw.attacking_strength * 19 + league_table["away"].GF * league_table["away"].MP / league_table["away"].GF.mean())/(19+league_table["away"].MP), 
            (away_sw.defensive_weakness * 19 + league_table["away"].GA * league_table["away"].MP / league_table["away"].GA.mean())/(19+league_table["away"].MP)
        ], 
        axis=1
        ).rename(
        columns={
            0: "attacking_strength",
            1: "defensive_weakness"
        }
        )
    ).to_dict(orient="index")

    GF_update = (GF_home * 380 + league_table["home"].GF.sum())/(380 + league_table["home"].MP.sum())
    GA_update = (GA_home * 380 + league_table["home"].GA.sum())/(380 + league_table["home"].MP.sum())
    
    return home_update, away_update, GF_update, GA_update
    

def update_league_table(matchweek, league_table):
    summary_table = league_table["summary"]
    home_table = league_table["home"]
    away_table = league_table["away"]
    
    for match in matchweek:
        home_team = match["home"]
        away_team = match["away"]
        home_goals = match["home_goals"]
        away_goals = match["away_goals"]
        GD = home_goals - away_goals

        home_points = 3 if GD > 0 else 1 if GD == 0 else 0
        away_points = 3 if GD < 0 else 1 if GD == 0 else 0
        
        home_line = [1, 1 if GD > 0 else 0, 1 if GD == 0 else 0, 1 if GD < 0 else 0, home_goals, away_goals, GD, home_points]
        away_line = [1, 1 if GD < 0 else 0, 1 if GD == 0 else 0, 1 if GD > 0 else 0, away_goals, home_goals, -GD, away_points]
        
        home_table.loc[home_team, :] += home_line
        away_table.loc[away_team, :] += away_line
        summary_table.loc[home_team, :] += home_line
        summary_table.loc[away_team, :] += away_line
    
    
    return {"summary": summary_table.sort_values(["Pts", "GD", "GF"], ascending=False),
           "home": home_table.sort_values(["Pts", "GD", "GF"], ascending=False),
           "away": away_table.sort_values(["Pts", "GD", "GF"], ascending=False)}

In [587]:
def simulate_season():
    season = []
    league_table = get_new_league_table()
    home_sw = home_sw_start.copy()
    away_sw = away_sw_start.copy()
    elo = elo_start.copy()
    GF = GF_home
    GA = GA_home
    for week in range(1, 39):
        matchweek, elo = simulate_matchweek(df_fixtures, week, home_sw, away_sw, elo, GF, GA)
        season.extend(matchweek)
        league_table = update_league_table(matchweek, league_table)
        home_sw, away_sw, GF, GA = update_sw(league_table, home_sw, away_sw, GF, GA)
        
    return {"all_matches": season, 
            "league_table": league_table,
            "home_sw": home_sw,
            "away_sw": away_sw,
            "elo": elo,
            "GF": GF,
            "GA": GA
           }

In [591]:
champions_no_elo = []
seasons_no_elo = []
for sim in range(10_000):
    season = simulate_season()
    seasons_no_elo.append(season)
        
    champion = season["league_table"]["summary"].index[0]
    champions_no_elo.append(champion)
    clear_output(wait=True)
    display(pd.Series(champions_no_elo, name="champion").value_counts())
    
with open("simulation_no_elo.pickle", "wb") as f:
    pickle.dump(seasons_no_elo, f)

champion
Arsenal            3545
Manchester City    2830
Liverpool          1693
Newcastle Utd       501
Aston Villa         267
Chelsea             242
Tottenham           175
Crystal Palace      139
Fulham              107
Manchester Utd       98
Leicester City       87
Brentford            69
Bournemouth          57
Everton              56
Brighton             40
Nott'ham Forest      25
West Ham             24
Ipswich Town         19
Wolves               15
Southampton          11
Name: count, dtype: int64